# Iterative identification of Boolean networks

Since **NORDic 2.8.0**, it is possible to update a gene regulatory network with new interactions, while keeping the other edges fixed. 

For illustration purposes, this vignette features a dummy example.

## Import

In [1]:
import NORDic
import subprocess
out = subprocess.check_output('python3 -m pip freeze | grep "NORDic"', shell=True).decode("utf-8").split("\n")[0]
out

'NORDic==2.8.0'

## Parameters

In [2]:
import pandas as pd

seednb=123456 # random seed
njobs=1 # number of jobs for the inference
limit=3 # maximum number of solutions to generate
max_iter=100 # maximum number of subsets of putative interactions to evaluate

## 1. Build the Prior Knowledge Network: identify interactions to keep and those which should be sampled

In [3]:
## Fixed interactions
solution_fixed = pd.DataFrame([], index=["A", "B", "C", "D"], columns=[1])
solution_fixed.loc["A"] = "!D" 
solution_fixed.loc["B"] = "A"
solution_fixed.loc["C"] = "!B"
solution_fixed.loc["D"] = "0"

solution_fixed

,1
A,!D
B,A
C,!B
D,0


In [4]:
## Putative interactions
net_putative = pd.DataFrame([], index=["regulated","regulator", "sign", "score"]).T
net_putative.loc[0] = ["C","D",1,1]
net_putative.loc[1] = ["A","C",-1,1]
net_putative.loc[2] = ["D","B",-1,1]
net_putative.loc[3] = ["B","D",-1,1]

net_putative

,regulated,regulator,sign,score
0,C,D,1,1
1,A,C,-1,1
2,D,B,-1,1
3,B,D,-1,1


In [5]:
## Define all genes present both in the fixed and putative interactions
all_gene_list = list(sorted(list(set(list(solution_fixed.index)+list(net_putative["regulated"])+list(net_putative["regulator"])))))
"#genes=%d" % len(all_gene_list)

'#genes=4'

## 2. Define the Dynamical Constraints: at least the initial state + final (attractor) state for each experiment

In [6]:
## dictionary of experiments with perturbation, initial state, and final attractor state
experiments = {}

initial = pd.DataFrame([], columns=["initial"], index=all_gene_list)

pert1 = {"A": 1}
final1 = pd.DataFrame([], columns=["final"], index=all_gene_list)
final1.loc['B'] = 1
final1.loc['C'] = 0
final1.loc['D'] = 0
experiments.update({"exp1": (pert1, initial, final1)})

pert2 = {"C": 0}
final2 = pd.DataFrame([], columns=["final"], index=all_gene_list)
final2.loc['D'] = 1
experiments.update({"exp2": (pert2, initial, final2)})

pert3 = {"D": 1}
final3 = pd.DataFrame([], columns=["final"], index=all_gene_list)
final3.loc['A'] = 0
final3.loc['B'] = 0
final3.loc['C'] = 1
experiments.update({"exp3": (pert3, initial, final3)})

pert4 = {"B": 1}
final4 = pd.DataFrame([], columns=["final"], index=all_gene_list)
final4.loc['A'] = 1
final4.loc['D'] = 0
final4.loc['C'] = 0
experiments.update({"exp4": (pert4, initial, final4)})

experiments

{'exp1': ({'A': 1},
    initial
  A     NaN
  B     NaN
  C     NaN
  D     NaN,
    final
  A   NaN
  B     1
  C     0
  D     0),
 'exp2': ({'C': 0},
    initial
  A     NaN
  B     NaN
  C     NaN
  D     NaN,
    final
  A   NaN
  B   NaN
  C   NaN
  D     1),
 'exp3': ({'D': 1},
    initial
  A     NaN
  B     NaN
  C     NaN
  D     NaN,
    final
  A     0
  B     0
  C     1
  D   NaN),
 'exp4': ({'B': 1},
    initial
  A     NaN
  B     NaN
  C     NaN
  D     NaN,
    final
  A     1
  B   NaN
  C     0
  D     0)}

## Run the iterative inference pipeline

In [7]:
from NORDic.NORDic_NI.iterative_network_identification import iterative_network_identification

net = iterative_network_identification(seednb, njobs, solution_fixed, net_putative, experiments, limit=limit, max_iter=max_iter)
net ## list of generated solutions

ipylab module is not installed, menus and toolbar are disabled.
Experiment exp4 with perturbation {'B': 1}: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 4/4 [00:00<00:00, 72.37it/s]



('network0.bnet', 0, 0, array([ True,  True,  True,  True]), [np.True_, np.True_, np.True_, np.True_])



Experiment exp4 with perturbation {'B': 1}: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 4/4 [00:00<00:00, 78.11it/s]



('network1.bnet', 1, 1, array([ True,  True, False,  True]), [np.True_, np.True_, np.True_, np.True_])



Experiment exp4 with perturbation {'B': 1}: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 4/4 [00:00<00:00, 80.27it/s]


('network2.bnet', 2, 2, array([False, False,  True,  True]), [np.True_, np.True_, np.True_, np.True_])

['network0.bnet', 'network1.bnet', 'network2.bnet']


['network0.bnet', 'network1.bnet', 'network2.bnet']

The prints show:
- the network name
- the number of current (valid) solutions available
- the total number of solutions found so far
- the putative interactions features in the network (True if present, False if absent, in the order in *net_putative*)
- whether each of the experiments in the *experiments* dictionary is predicted by the network (True) or not (False).
Let us have a look at those solutions:

In [8]:
for solution_fname in net:
    network = pd.read_csv(solution_fname, index_col=0)
    print(solution_fname)
    print(network)
    print("____________")

network0.bnet
   (!D)|(!C)
A           
B   (A)|(!D)
C   (!B)|(D)
D         !B
____________
network1.bnet
   (!D)|(!C)
A           
B   (A)|(!D)
C   (!B)|(D)
D          0
____________
network2.bnet
          !D
A           
B   (A)|(!D)
C         !B
D         !B
____________


**NORDic** is an open-source package which allows to focus on a network-oriented approach to identify regulatory mechanisms linked to a disease, master regulators:

+ **NORDic NI** identifies a disease-associated gene regulatory network (as a Boolean network) with its dynamics from several biological sources.
+ **NORDic PMR** detects master regulators in a Boolean network.

Other approaches regarding the simulation of drug treatment and drug repurposing are also a part of **NORDic**:

+ **NORDic DS** (since version **2.0.0**) scores the effect of a treatment on a patient (the higher the score, the most promising the treatment) based on a Boolean network.
+ **NORDic DR** (since version **2.0.0**) uses the routine in **NORDic DS** and a bandit algorithm to adaptively test treatments and perform drug repurposing.

Corresponding notebooks can be found at [this page](https://github.com/clreda/NORDic/tree/main/notebooks).

In [9]:
from subprocess import call
call("rm -rf "+file_folder, shell=True)
call("rm -f experiments.csv network.tsv", shell=True)

NameError: name 'file_folder' is not defined